In [0]:
#Generate Streaming Source
from pyspark.sql.functions import *

In [0]:
stream_df=(
    spark.readStream
    .format("rate")
    .option("rowsPerSecond",5)
    .load()
)

In [0]:
#Check Schema
stream_df.printSchema()


In [0]:
#Create Financial Transactions
finance_stream_df=(
    stream_df
    .withColumn("transaction_id",col("value"))
    .withColumn("customer_id",(rand()*5000).cast("int"))
    .withColumn("transaction_amount",
                round(rand()*5000+10,2))
    .withColumn("transaction_type",
        when(rand()<0.25,"UPI")
        .when(rand()<0.50,"NEFT")
        .when(rand()<0.75,"Credit Card")
        .otherwise("Debit Card")
        )
    .withColumn("is_fraud",
         when(rand() <0.03,1).otherwise(0)
         )
    )


In [0]:
spark.sql("SHOW CATALOGS").show(truncate=False)

In [0]:
spark.sql("SHOW SCHEMAS").show(truncate=False)

In [0]:
spark.sql("""
SELECT current_catalog(),
       current_schema()
""").show()

In [0]:
spark.sql("SHOW VOLUMES").show(truncate=False)

In [0]:
query = (finance_stream_df
    .writeStream
    .format("memory")
    .queryName("finance_transactions_view")
    .outputMode("append")
    .trigger(availableNow=True)
    .option("checkpointLocation", "/Volumes/workspace/default/finance_volume/checkpoints/finance_stream_view")
    .start())

query.awaitTermination()
display(spark.table("finance_transactions_view"))

In [0]:
#Write Stream to Delta Table
query = (
    finance_stream_df.writeStream
    .format("delta")
    .outputMode("append")
    .trigger(availableNow=True)
    .option("checkpointLocation",
            "/Volumes/workspace/default/finance_volume/checkpoints/finance_stream")
    .toTable("stream_transactions")
)

In [0]:
#Verify Data
display(
    spark.sql("""
    SELECT *
    FROM stream_transactions
    ORDER BY timestamp DESC
    LIMIT 20
    """)
)

In [0]:
#Create Real-Time Fraud Stream
fraud_stream = spark.readStream.table(
    "stream_transactions"
)

In [0]:
fraud_stream_filtered = fraud_stream.filter(
    col("is_fraud") == 1
)

In [0]:
#Save Fraud Alerts
fraud_query = (
    fraud_stream_filtered.writeStream
    .format("delta")
    .outputMode("append")
    .trigger(availableNow=True)
    .option(
        "checkpointLocation",
        "/Volumes/workspace/default/finance_volume/checkpoints/fraud_stream"
    )
    .toTable("stream_fraud_alerts")
)

In [0]:
#Verify Fraud Alerts
display(
    spark.sql("""
    SELECT *
    FROM stream_fraud_alerts
    ORDER BY timestamp DESC
    LIMIT 20
    """)
)

In [0]:
#Streaming Metrics
#Total fraud transactions
spark.sql("""
SELECT COUNT(*) AS fraud_count
FROM stream_fraud_alerts
""")

In [0]:
#Average fraud amount
spark.sql("""
SELECT AVG(transaction_amount)
FROM stream_fraud_alerts
""")